# Offline-First Volatility Regime Walkthrough

## 1) Research question and offline contract

We study one question: **do latent regimes from option-implied volatility-surface features improve forecasts of future realized volatility** compared with simple baselines?

This notebook is strictly offline at runtime:

- It reads only tracked Parquet and JSON sidecars from `data/raw/`.
- It does not require ClickHouse for normal execution.
- It uses package functions from `volatility_regimes` so the notebook mirrors the same implementation used by the CLI modules.

- Notebook demo window defaults to `2022-01-03` through `2024-12-31` for faster, comfortable iteration.

Throughout the notebook, we define each symbol before use. When we write formulas, we use: $P_t$ for close price at day $t$, $r_t$ for log return, $h$ for forecast horizon in trading days, and $A$ for annualization factor (default $252$).

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import tomllib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from volatility_regimes.data_access.loader import load_daily_prices
from volatility_regimes.data_access.loader import load_options
from volatility_regimes.features.surface import extract_features
from volatility_regimes.features.surface import select_feature_columns
from volatility_regimes.regimes.latent_models import fit_gmm
from volatility_regimes.regimes.latent_models import fit_hmm
from volatility_regimes.regimes.latent_models import order_regimes_by_volatility
from volatility_regimes.regimes.latent_models import standardize_features
from volatility_regimes.descriptive.analytics import compute_realized_vol
from volatility_regimes.walkforward.targets import build_forward_targets
from volatility_regimes.walkforward.splits import build_expanding_window_splits
from volatility_regimes.walkforward.models import forecast_atm_iv
from volatility_regimes.walkforward.models import forecast_linear_features
from volatility_regimes.walkforward.models import forecast_regime_mean
from volatility_regimes.walkforward.models import forecast_trailing_realized_vol
from volatility_regimes.walkforward.reporting import summarize_metrics
from volatility_regimes.walkforward.engine import _apply_forward_target_embargo
from volatility_regimes.walkforward.engine import _build_price_date_positions

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

config_path = PROJECT_ROOT / "config.toml"
with config_path.open("rb") as file_handle:
    config = tomllib.load(file_handle)

data_config = config["data"]
cache_config = config["cache"]
feature_config = config["features"]
analysis_config = config["analysis"]

symbol = "SPX"
demo_start_date = "2022-01-03"
demo_end_date = "2024-12-31"
start_date = demo_start_date
end_date = demo_end_date
annualization = int(analysis_config["annualization_factor"])
horizon = int(analysis_config["realized_vol_window"])

print("Project root:", PROJECT_ROOT)
print("Offline raw dir:", PROJECT_ROOT / cache_config["required_dir"])
print("Symbol:", symbol, "Date range:", start_date, "to", end_date)

## 2) Raw input files and schema expectations

The offline contract requires one options dataset and one prices dataset per symbol, each with:

1. a Parquet file containing rows
2. a metadata sidecar (`.metadata.json`) containing schema, row count, and date coverage

We first confirm those files exist and inspect sidecar metadata so the data-loading assumptions are explicit before any modeling.

In [ ]:
raw_dir = PROJECT_ROOT / str(cache_config["required_dir"])
required_files = [
    raw_dir / "options_spx.parquet",
    raw_dir / "options_spx.metadata.json",
    raw_dir / "options_ndx.parquet",
    raw_dir / "options_ndx.metadata.json",
    raw_dir / "prices_spx.parquet",
    raw_dir / "prices_spx.metadata.json",
    raw_dir / "prices_ndx.parquet",
    raw_dir / "prices_ndx.metadata.json",
]

missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Missing required raw files: {missing_files}")

metadata_rows = []
for metadata_path in sorted(raw_dir.glob("*.metadata.json")):
    with metadata_path.open("r", encoding="utf-8") as metadata_handle:
        payload = json.load(metadata_handle)
    metadata_rows.append(
        {
            "file": metadata_path.name,
            "dataset": payload.get("dataset"),
            "symbol": payload.get("symbol"),
            "row_count": payload.get("row_count"),
            "start": payload.get("date_coverage", {}).get("start"),
            "end": payload.get("date_coverage", {}).get("end"),
        }
    )

metadata_table = pd.DataFrame(metadata_rows).sort_values(["dataset", "symbol"]).reset_index(drop=True)
metadata_table

## 3) Price series, log returns, and forward realized-volatility target

Let $P_t$ be close price at day $t$. Define daily log return as

$$
r_t = \log\left(rac{P_t}{P_{t-1}}ight).
$$

For horizon $h$ and annualization factor $A$, forward realized volatility is

$$
\mathrm{RV}_t(h) = \mathrm{std}(r_{t+1}, \ldots, r_{t+h})\sqrt{A}.
$$

We compute this target and inspect magnitudes to catch obvious scale errors early.

In [ ]:
prices = load_daily_prices(
    symbol=symbol,
    start_date=start_date,
    end_date=end_date,
    cache_config=cache_config,
)

close_series = prices.set_index("date")["close"].sort_index()
log_returns = np.log(close_series / close_series.shift(1))
realized_vol = compute_realized_vol(
    prices=prices,
    window=horizon,
    annualization=annualization,
)

target_preview = pd.DataFrame(
    {
        "close": close_series,
        "log_return": log_returns,
        "forward_realized_vol": realized_vol,
    }
).dropna().head(8)

print("Prices rows:", len(prices))
print("Log-return mean (daily):", float(log_returns.mean()))
print("Forward RV mean (annualized):", float(realized_vol.dropna().mean()))
target_preview

## 4) Option-chain slices and feature engineering logic

From each trade date, we derive a compact volatility-surface representation:

- near-term and mid-term ATM implied volatility levels
- skew terms
- butterfly (curvature) terms
- term slope

These features become model inputs for both latent regime fitting and walk-forward forecasting.

In [ ]:
options = load_options(
    symbol=symbol,
    start_date=start_date,
    end_date=end_date,
    delta_min=float(feature_config["delta_min"]),
    delta_max=float(feature_config["delta_max"]),
    cache_config=cache_config,
)

features = extract_features(
    options=options,
    near_dte_min=int(feature_config["near_term_dte_min"]),
    near_dte_target=int(feature_config["near_term_dte_target"]),
    near_dte_max=int(feature_config["near_term_dte_max"]),
    mid_dte_min=int(feature_config["mid_term_dte_min"]),
    mid_dte_target=int(feature_config["mid_term_dte_target"]),
    mid_dte_max=int(feature_config["mid_term_dte_max"]),
    wing_delta=float(feature_config["wing_delta"]),
    min_strikes=int(feature_config["min_strikes_per_side"]),
).dropna()

print("Options rows:", len(options))
print("Feature rows:", len(features))
features.head(6)

## 5) Regime model intuition and ordered-state labeling

We fit two latent-state model families:

1. **Gaussian Mixture Model (GMM)**: clusters rows independently after feature scaling.
2. **Hidden Markov Model (HMM)**: adds transition dynamics between latent states over time.

We then reorder state labels from lower-volatility to higher-volatility regimes using ATM volatility as the ordering anchor so regime IDs are interpretable.

In [ ]:
model_features = select_feature_columns(features=features, feature_set="atm_term").tail(800)
standardized_matrix, _ = standardize_features(model_features.to_numpy(dtype=float))

gmm_labels_raw, bic_scores, best_k, _ = fit_gmm(
    feature_matrix=standardized_matrix,
    min_k=2,
    max_k=3,
)
hmm_labels_raw, _, _ = fit_hmm(
    feature_matrix=standardized_matrix,
    n_states=best_k,
    n_iter=50,
    n_restarts=2,
)

gmm_labels = order_regimes_by_volatility(gmm_labels_raw, standardized_matrix, atm_iv_col_idx=0)
hmm_labels = order_regimes_by_volatility(hmm_labels_raw, standardized_matrix, atm_iv_col_idx=0)

regime_counts = pd.DataFrame(
    {
        "gmm_count": pd.Series(gmm_labels).value_counts().sort_index(),
        "hmm_count": pd.Series(hmm_labels).value_counts().sort_index(),
    }
)

print("BIC scores:", bic_scores)
print("Selected K:", best_k)
regime_counts

## 6) Walk-forward design, embargo rule, and benchmark set

We now switch from descriptive analysis to out-of-sample forecasting.

For each split:

- training dates are strictly before test dates
- targets are built as forward realized volatility
- an **embargo** removes train rows whose target windows overlap the test window

The benchmark family includes ATM IV, trailing realized volatility, linear features, and GMM regime-mean forecasts.

In [ ]:
targets = build_forward_targets(
    prices=prices,
    features=features,
    horizon=horizon,
    annualization=annualization,
)

walkforward_features = select_feature_columns(features, "atm_term").tail(900)
walkforward_targets = targets.reindex(walkforward_features.index)
combined = walkforward_features.join(walkforward_targets[["realized_vol"]]).dropna()

splits = build_expanding_window_splits(
    dates=pd.DatetimeIndex(combined.index),
    min_train_size=700,
    step_size=5,
)

first_split = splits[0]
price_date_positions = _build_price_date_positions(prices)
embargoed_train_index = _apply_forward_target_embargo(
    train_index=first_split.train_index,
    test_index=first_split.test_index,
    horizon=horizon,
    price_date_positions=price_date_positions,
)

print("Combined rows:", len(combined))
print("Train rows before embargo:", len(first_split.train_index))
print("Train rows after embargo:", len(embargoed_train_index))
print("Test rows:", len(first_split.test_index))

## 7) Model fitting and forecast generation

For a small out-of-sample slice, we build one forecast panel with four models:

- `atm_iv`
- `trailing_realized_vol`
- `linear_features`
- `gmm_regime_mean`

This demonstrates the same forecast API used by the walk-forward engine without launching a subprocess.

In [ ]:
train_features = combined.loc[embargoed_train_index, walkforward_features.columns]
train_target = combined.loc[embargoed_train_index, "realized_vol"]
test_features = combined.loc[first_split.test_index, walkforward_features.columns]
test_target = combined.loc[first_split.test_index, "realized_vol"]

# trailing realized volatility benchmark aligned to evaluation index
rolling_std = log_returns.rolling(window=horizon).std() * np.sqrt(float(annualization))
trailing_realized_vol = rolling_std.reindex(test_features.index)

forecast_rows = []
for test_date, test_row in test_features.head(6).iterrows():
    actual_value = float(test_target.loc[test_date])
    atm_prediction = forecast_atm_iv(test_row)
    trailing_prediction = forecast_trailing_realized_vol(trailing_realized_vol, test_date)
    linear_prediction = forecast_linear_features(train_features, train_target, test_row)
    gmm_result = forecast_regime_mean(
        train_features=train_features,
        train_target=train_target,
        test_row=test_row,
        model_type="gmm",
        min_k=2,
        max_k=3,
    )

    forecast_rows.extend(
        [
            {"symbol": symbol, "horizon": horizon, "feature_set": "atm_term", "date": test_date, "model_name": "atm_iv", "prediction": float(atm_prediction), "actual": actual_value},
            {"symbol": symbol, "horizon": horizon, "feature_set": "atm_term", "date": test_date, "model_name": "trailing_realized_vol", "prediction": float(trailing_prediction), "actual": actual_value},
            {"symbol": symbol, "horizon": horizon, "feature_set": "atm_term", "date": test_date, "model_name": "linear_features", "prediction": float(linear_prediction), "actual": actual_value},
            {"symbol": symbol, "horizon": horizon, "feature_set": "atm_term", "date": test_date, "model_name": "gmm_regime_mean", "prediction": float(gmm_result["prediction"]), "actual": actual_value},
        ]
    )

forecast_panel = pd.DataFrame(forecast_rows)
forecast_panel.head(12)

## 8) Metric summary and interpretation

We aggregate root mean squared error (RMSE), mean absolute error (MAE), and out-of-sample $R^2$ relative to the ATM benchmark.

We also include the **variance risk premium** (defined as ATM IV minus forward realized volatility) in interpretation context: strong predictive performance in one window does not prove a stable structural edge.

In [ ]:
metric_summary = summarize_metrics(forecast_panel)

output_dir = PROJECT_ROOT / "outputs" / "figures" / "walkforward"
output_dir.mkdir(parents=True, exist_ok=True)
plot_path = output_dir / "notebook_metric_demo.png"

plot_frame = metric_summary.sort_values("rmse")
fig, axis = plt.subplots(figsize=(10, 4), constrained_layout=True)
axis.bar(plot_frame["model_name"], plot_frame["rmse"], color="#2c3e50")
axis.set_title("Walk-forward RMSE (Notebook Demo Slice)")
axis.set_ylabel("RMSE")
axis.set_xlabel("Model")
axis.grid(alpha=0.3, axis="y")
fig.savefig(plot_path, dpi=160)
plt.close(fig)

print("Saved metric plot:", plot_path)
metric_summary

## 9) Failure modes, limitations, and extensions

This notebook demonstrates an offline research pipeline, not a final trading system.

Main limitations:

1. Small model families can miss nonlinear structure.
2. Regime labels are statistical constructs, not causal states.
3. Results depend on horizon choice, feature subset, and sample window.
4. HMM fitting can be numerically unstable in short windows or weakly separated regimes.

**What Else?**

- Extend to probability-weighted regime forecasts instead of hard-state mapping.
- Compare against richer baselines (for example, regularized linear models).
- Add uncertainty intervals for metric differences across models.
- Stress-test sensitivity to annualization, embargo width, and feature set definitions.

**TL;DR**

The project runs fully offline from `data/raw/`, builds volatility-surface features, fits latent regimes, and evaluates out-of-sample forecasts with leakage controls in one reproducible workflow.

In [ ]:
output_snapshot = {
    "descriptive_reports_dir": str(PROJECT_ROOT / "outputs" / "reports" / "descriptive"),
    "descriptive_figures_dir": str(PROJECT_ROOT / "outputs" / "figures" / "descriptive"),
    "walkforward_reports_dir": str(PROJECT_ROOT / "outputs" / "reports" / "walkforward"),
    "walkforward_figures_dir": str(PROJECT_ROOT / "outputs" / "figures" / "walkforward"),
    "forecast_rows_demo": int(len(forecast_panel)),
    "metric_rows_demo": int(len(metric_summary)),
}
output_snapshot